<font size=10>**DATA EXPLORATION & PREPROCESSING**</font> <a class="anchor" id='title'></a> 

**Bachelor's in Data Science - NOVA IMS (25/26)**

<font color='#BFD72' size=5>**RESEARCH QUESTION**: </font><font size=5>*Which companies have a dominant position within specific municipalities?*</font> 

**Data**: 
- [*Portal BASE*](https://www.base.gov.pt/Base4/pt/pesquisa/?type=contratos&texto=&adjudicante=&adjudicataria=&tipo=2&tipocontrato=0&cpv=&aqinfo=&desdeprazoexecucao=&ateprazoexecucao=&sel_price=price_11&desdeprecocontrato=&ateprecocontrato=&desdeprecoefectivo=&ateprecoefectivo=&sel_date=date_11&desdedatacontrato=2023-01-01&atedatacontrato=2026-03-31&desdedatapublicacao=&atedatapublicacao=&desdedatafecho=&atedatafecho=&pais=0&distrito=0&concelho=0)

- [*Treated Datasets*](https://dados.gov.pt/pt/datasets/contratos-publicos-portal-base-impic-contratos-de-2012-a-2026/#/resources)

**Group B**
- Beatriz Marques 20231605
- Maria Inês Santos 20231630
- Luís Soeiro 20211536
- Rodrigo Silva 20231602

<font color='#BFD72' size=6>**TABLE OF CONTENTS**</font> <a class="anchor" id='toc'></a>  
- [1. Imports](#1-imports)  
- [2. Data Integration](#2-data-integration)  
- [3. Data Preprocessing](#3-data-preprocessing)  
    - [3.1 Duplicates](#31-duplicates)
    - [3.2 Missing Values](#32-missing-values)
    - [3.3 Preprocessing Per Column](#33-preprocessing-per-column)
        - [3.3.1 Date Columns](#331-date-columns)
        - [3.3.2 Tipo de Procedimento](#332-tipo-de-procedimento)
        - [3.3.3 Tipo de Contrato](#333-tipo-de-contrato)
        - [3.3.4 Concorrentes](#334-concorrentes)
        - [3.3.5 Adjudicante & Adjudicatário](#335-adjudicante--adjudicatário)
        - [3.3.6 Local de Execução](#336-local-de-execução)
        - [3.3.7 Price Columns](#337-price-columns)
        - [3.3.8 CPV](#338-cpv)
    - [3.4 Final Data](#34-final-data)
- [4. Export Preprocessed Data](#4-export-preprocessed-data)

# <font color='#BFD72F' size=6>**1. Imports**</font> <a class="anchor" id="1"></a>

[Back to TOC](#toc)

In [1]:
import warnings
%load_ext autoreload
%autoreload 2

warnings.filterwarnings('ignore')

In [2]:
import sys
import os

# Get the absolute path of the source_code folder
source_code_path = os.path.abspath('../source')

# Add the source_code folder to sys.path
if source_code_path not in sys.path:
    sys.path.append(source_code_path)

In [3]:
import subprocess, sys, importlib
import pandas as pd
import plotly.express as px
import re
import plotly.graph_objects as go

try:
    import openpyxl
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "openpyxl"])
    importlib.invalidate_caches()
    import openpyxl
import os
import shutil

# <font color='#BFD72F' size=6>**2. Data Integration**</font> <a class="anchor" id="2"></a>
  
[Back to TOC](#toc)

$\rightarrow$ **Contract Timeframe**: From January 1, 2023 to April 25, 2026

In [4]:
# MERGE DATASETS
paths = {
#    "2023_part01": "../data/contratos2023_part01.csv",
#    "2023_part02": "../data/contratos2023_part02.csv",
#    "2023_part03": "../data/contratos2023_part03.csv",
#    "2024_part01": "../data/contratos2024_part01.csv",
#    "2024_part02": "../data/contratos2024_part02.csv",
#    "2024_part03": "../data/contratos2024_part03.csv",
    "2025_part01": "../data/contratos2025_part01.csv",
    "2025_part02": "../data/contratos2025_part02.csv",
    "2025_part03": "../data/contratos2025_part03.csv",
    "2026": "../data/contratos2026.csv",
}

datasets = {}

merged_dataset = pd.DataFrame()

for year, path in paths.items():
    print(f"Loading dataset for {year} from {path}...")
    datasets[year] = pd.read_csv(path)
    print(f"Dataset for {year} loaded successfully with shape {datasets[year].shape}.")
    merged_dataset = pd.concat([merged_dataset, datasets[year]], ignore_index=True)

print(f"Merged dataset created with shape {merged_dataset.shape}.")

Loading dataset for 2025_part01 from ../data/contratos2025_part01.csv...
Dataset for 2025_part01 loaded successfully with shape (81131, 35).
Loading dataset for 2025_part02 from ../data/contratos2025_part02.csv...
Dataset for 2025_part02 loaded successfully with shape (81132, 35).
Loading dataset for 2025_part03 from ../data/contratos2025_part03.csv...
Dataset for 2025_part03 loaded successfully with shape (81132, 35).
Loading dataset for 2026 from ../data/contratos2026.csv...
Dataset for 2026 loaded successfully with shape (69171, 35).
Merged dataset created with shape (312566, 35).


In [5]:
merged_dataset.info()

<class 'pandas.DataFrame'>
RangeIndex: 312566 entries, 0 to 312565
Data columns (total 35 columns):
 #   Column                    Non-Null Count   Dtype  
---  ------                    --------------   -----  
 0   idcontrato                312566 non-null  int64  
 1   nAnuncio                  48447 non-null   str    
 2   TipoAnuncio               48447 non-null   str    
 3   idINCM                    48447 non-null   float64
 4   tipoContrato              312566 non-null  str    
 5   idprocedimento            312566 non-null  int64  
 6   tipoprocedimento          312566 non-null  str    
 7   objectoContrato           312565 non-null  str    
 8   descContrato              312566 non-null  str    
 9   adjudicante               312566 non-null  str    
 10  adjudicatarios            312487 non-null  str    
 11  dataPublicacao            312566 non-null  str    
 12  dataCelebracaoContrato    311273 non-null  str    
 13  precoContratual           312566 non-null  float64
 14 

$\rightarrow$**Columns To Keep**:

**Identifiers & Contract Info**

| column name | |
|--- | --- |
| idcontrato | |
| tipoContrato | |
| tipoFimContrato | |
| CPV | |
| tipoprocedimento | |

**Entities**

| column name | |
|--- | --- |
| adjudicante | |
| adjudicatarios | |
| concorrentes | |

**Financial Variables**

| column name | |
|--- | --- |
| precoBaseProcedimento | |
| precoContratual | |
| PrecoTotalEfetivo | |

**Location** 

| column name | |
|--- | --- |
| LocalExecucao | |

**Dates** 

| column name | |
|--- | --- |
| dataDecisaoAdjudicacao | |
| dataCelebracaoContrato | |
| dataPublicacao | |
| dataFechoContrato | |

In [6]:
cols_to_keep = [
    'idcontrato', 'tipoContrato', 'tipoFimContrato', 'CPV', 'tipoprocedimento',
    'adjudicante', 'adjudicatarios', 'concorrentes', 
    'precoBaseProcedimento', 'precoContratual', 'PrecoTotalEfetivo', 
    'LocalExecucao', 
    "dataDecisaoAdjudicacao", "dataCelebracaoContrato", "dataPublicacao", "dataFechoContrato"
]

subset = merged_dataset[cols_to_keep]

In [7]:
print("There are {} Public Entities.".format(subset['adjudicante'].nunique()))
print("There are {} Companies.".format(subset['adjudicatarios'].nunique()))

print("So, in total our analysis contains {} Nodes.".format(
    subset['adjudicatarios'].nunique() + 
    subset['adjudicante'].nunique()))

There are 7482 Public Entities.
There are 80949 Companies.
So, in total our analysis contains 88431 Nodes.


In [8]:
print("There are {} Public Entities.".format(subset['adjudicante'].nunique()))
print("There are {} Companies.".format(subset['adjudicatarios'].nunique()))

print("So, in total our analysis contains {} Nodes.".format(
    subset['adjudicatarios'].nunique() + 
    subset['adjudicante'].nunique()))

There are 7482 Public Entities.
There are 80949 Companies.
So, in total our analysis contains 88431 Nodes.


In [9]:
display(subset.iloc[0])

idcontrato                                                         11103352
tipoContrato                                  Empreitadas de obras públicas
tipoFimContrato                                                         NaN
CPV                                     45233120-6 - Construção de estradas
tipoprocedimento                                           Concurso público
adjudicante                  503933813 - Infraestruturas de Portugal, S. A.
adjudicatarios                                              500326517 - CJR
concorrentes              500070210-Conduril – Engenharia, S.A.\r\n50007...
precoBaseProcedimento                                            18000000.0
precoContratual                                                 14665084.64
PrecoTotalEfetivo                                                       0.0
LocalExecucao             Portugal, Porto, Baião\r\nPortugal, Porto, Mar...
dataDecisaoAdjudicacao                                           2024-11-08
dataCelebrac

# <font color='#BFD72F' size=6>**3. Data Preprocessing**</font> <a class="anchor" id="3"></a>
  
[Back to TOC](#toc)

## <font size=6>**3.1 Duplicates**</font> <a class="anchor" id="3.1"></a>
  
[Back to TOC](#toc)

In [9]:
# Check duplicated rows
duplicated_rows = subset.duplicated()
print(f"Number of duplicated rows: {duplicated_rows.sum()}")

Number of duplicated rows: 1


In [10]:
# dropping duplicated rows
subset = subset.drop_duplicates()

## <font size=6>**3.2 Missing Values**</font> <a class="anchor" id="3.2"></a>
  
[Back to TOC](#toc)

In [11]:
total = len(subset)
missing_counts = subset.isnull().sum()
missing_pct = (missing_counts / total * 100).round(2)

missing_df = pd.DataFrame({
    'missing_count': missing_counts,
    'missing_pct': missing_pct
}).sort_values('missing_pct', ascending=False)

# show only columns with any missing values
missing_df

,missing_count,missing_pct
tipoFimContrato,273253,87.42
dataFechoContrato,271951,87.01
concorrentes,181045,57.92
dataDecisaoAdjudicacao,1293,0.41
dataCelebracaoContrato,1293,0.41
LocalExecucao,944,0.30
adjudicatarios,79,0.03
idcontrato,0,0.00
tipoprocedimento,0,0.00
adjudicante,0,0.00


In [12]:
# rows with missing values except for 'concorrentes', 'dataFechoContrato' and 'tipoFimContrato'
subset = subset.dropna(subset=[col for col in subset.columns if col not in ['concorrentes', 'dataFechoContrato', 'tipoFimContrato']])

In [13]:
print("Initial Number of Contracts: {}".format(merged_dataset.shape[0]))
print("Number of Contracts Now: {}".format(subset.shape[0]))
print("Percentage of Deleted Contracts: {}%".format(round((1 - (subset.shape[0]/merged_dataset.shape[0]))*100, 2)))

Initial Number of Contracts: 312566
Number of Contracts Now: 311192
Percentage of Deleted Contracts: 0.44%


## <font size=6>**3.3 Preprocessing Per Column**</font> <a class="anchor" id="3.3"></a>
  
[Back to TOC](#toc)

### <font size=6>3.3.1 Date Columns</font> <a class="anchor" id="3.3.1"></a>
  
[Back to TOC](#toc)

In [14]:
# date type conversion    
subset["dataPublicacao"] = pd.to_datetime(subset["dataPublicacao"], errors='coerce')
subset["dataCelebracaoContrato"] = pd.to_datetime(subset["dataCelebracaoContrato"], errors='coerce')
subset["dataDecisaoAdjudicacao"] = pd.to_datetime(subset["dataDecisaoAdjudicacao"], errors='coerce')
subset["dataFechoContrato"] = pd.to_datetime(subset["dataFechoContrato"], errors='coerce')

In [15]:
dfs = []

date_cols = {
    'dataDecisaoAdjudicacao': 'Decision',
    'dataCelebracaoContrato': 'Celebration',
    'dataFechoContrato': 'Closure'
}

for col, label in date_cols.items():
    df = (
        subset
        .dropna(subset=[col])
        .assign(month=subset[col].dt.to_period('M').dt.to_timestamp())
        .groupby('month')
        .size()
        .reset_index(name='number_of_contracts')
    )
    
    df['type'] = label
    dfs.append(df)

final_df = pd.concat(dfs)

fig = px.line(
    final_df,
    x='month',
    y='number_of_contracts',
    color='type',
    title='Number of Contracts by Month (Different Dates)',
    labels={
        'month': 'Month',
        'number_of_contracts': 'Number of Contracts',
        'type': 'Date Type'
    }
)

fig.show()

In [16]:
# compute difference in months between celebration and closure, then plot histogram
months_df = subset.dropna(subset=['dataCelebracaoContrato','dataFechoContrato']).copy()
s = months_df['dataCelebracaoContrato']
e = months_df['dataFechoContrato']
months_df['months_diff'] = (e.dt.year - s.dt.year) * 12 + (e.dt.month - s.dt.month) + (e.dt.day - s.dt.day) / 30.0

fig_months = px.histogram(
    months_df,
    x='months_diff',
    nbins=100,
    title='Months between Celebration and Closure',
    labels={'months_diff': 'Months difference', 'count': 'Number of Contracts'}
)
fig_months.update_xaxes(range=[float(months_df['months_diff'].min()), float(months_df['months_diff'].max())])
fig_months.show()


In [17]:
# cut where closure date is before celebration date -> inconsistent data that should be removed
subset = subset[subset['dataFechoContrato'] >= subset['dataCelebracaoContrato']]

### <font size=6>3.3.2 Tipo de Procedimento</font> <a class="anchor" id="3.3.2"></a>
  
[Back to TOC](#toc)

$\rightarrow$ **tipoprocedimento**: Concurso público

In [18]:
subset = subset[subset['tipoprocedimento'] == 'Concurso público']
subset.drop(columns=['tipoprocedimento'], inplace=True)

In [19]:
subset.shape

(2845, 15)

### <font size=6>3.3.3 Tipo de Contrato</font> <a class="anchor" id="3.3.3"></a>
  
[Back to TOC](#toc)

In [20]:
subset['tipoContrato'] = subset['tipoContrato'].astype(str).str.replace(r'[\r\n]+', ' | ', regex=True).str.strip()

In [21]:
subset['tipoContrato'].value_counts()

tipoContrato
Aquisição de bens móveis                                    1837
Aquisição de serviços                                        590
Empreitadas de obras públicas                                294
Locação de bens móveis                                       101
Aquisição de bens móveis | Aquisição de serviços              10
Aquisição de serviços | Locação de bens móveis                 8
Aquisição de bens móveis | Locação de bens móveis              1
Empreitadas de obras públicas | Locação de bens móveis         1
Aquisição de bens móveis | Empreitadas de obras públicas       1
Aquisição de serviços | Outros                                 1
Concessão de serviços públicos                                 1
Name: count, dtype: int64

In [22]:
subset_counts = subset['tipoContrato'].value_counts().reset_index()
subset_counts.columns = ['tipoContrato', 'count']

# Get top 10
top_10 = subset_counts.head(10)

fig = px.bar(
    top_10,
    x='tipoContrato',
    y='count',
    title='Number of Contracts by Contract Type',
    labels={'tipoContrato': 'Contract Type', 'count': 'Number of Contracts'}
)

fig.show()

### <font size=6>3.3.4 Concorrentes</font> <a class="anchor" id="3.3.4"></a>
  
[Back to TOC](#toc)

In [23]:
subset['concorrentes'] = (
    subset['concorrentes']
    .astype(str)
    # replace line breaks with separator
    .str.replace(r'[\r\n]+', ' | ', regex=True)
    # remove excessive spaces
    .str.replace(r'\s+', ' ', regex=True)
    # remove trailing numbers (like " 102", " 89", etc.)
    .str.replace(r'\s+\d+\s*$', '', regex=True)
    # final trim
    .str.strip()
)

In [24]:
# number of competitors per contract
subset['nr_concorrentes'] = subset['concorrentes'].apply(
    lambda x: len([i for i in re.split(r'\s*\|\s*', x) if i]) 
    if isinstance(x, str) else 0
)

fig = px.histogram(
    subset,
    x='nr_concorrentes',
    nbins=30,
    title='Histogram of Number of Competitors'
)

fig.show()

### <font size=6>3.3.5 Adjudicante & Adjudicatário</font> <a class="anchor" id="3.3.5"></a>
  
[Back to TOC](#toc)

In [25]:
# extract contribuinte numbers from adjudicante and adjudicatarios
subset['contribuinte_adjudicante'] = subset['adjudicante'].str.extract(r'(\d{9})')
subset['adjudicante'] = subset['adjudicante'].str.replace(r'\s*\d{9}\s* - ', '', regex=True).str.strip()

subset['contribuinte_adjudicatarios'] = subset['adjudicatarios'].str.extract(r'(\d{9})')
subset['adjudicatarios'] = subset['adjudicatarios'].str.replace(r'\s*\d{9}\s* - ', '', regex=True).str.strip()

### <font size=6>3.3.6 Local de Execução</font> <a class="anchor" id="3.3.6"></a>
  
[Back to TOC](#toc)

$\rightarrow$ **district**: Lisboa

In [26]:
# LocalExecucao
subset['LocalExecucao'] = subset['LocalExecucao'].fillna('')

subset['LocalExecucao'] = (
    subset['LocalExecucao']
    # replace line breaks with separator
    .str.replace(r'[\r\n]+', ' | ', regex=True)
    # normalize spaces
    .str.replace(r'\s+', ' ', regex=True)
    .str.strip()
    # remove duplicates inside each cell
    .apply(lambda x: ' | '.join(dict.fromkeys(x.split(' | '))) if x else x)
)

first_location = subset['LocalExecucao'].str.split(' \| ', expand=False).str[0]

split_cols = first_location.str.split(', ', expand=True)

split_cols = split_cols.reindex(columns=[0, 1, 2])
split_cols.columns = ['country', 'district', 'city']

subset[['country', 'district', 'city']] = split_cols

for col in ['country', 'district', 'city']:
    subset[col] = subset[col].replace(r'^\s*$', pd.NA, regex=True)

# handle inconsistent structures
n_parts = first_location.str.split(', ').str.len()

# if only 1 part - it's country
subset.loc[n_parts == 1, ['district', 'city']] = pd.NA

# if 2 parts - assume country + district
subset.loc[n_parts == 2, 'city'] = pd.NA

In [27]:
subset[['country', 'district', 'city']].value_counts(dropna=False)

country   district                    city                  
Portugal  NaN                         NaN                       660
          Lisboa                      Lisboa                    503
          Setúbal                     Barreiro                  186
          Porto                       Porto                     116
          Coimbra                     Coimbra                    80
                                                               ... 
          Viseu                       Sernancelhe                 1
          Região Autónoma da Madeira  Ponta do Sol                1
          Região Autónoma dos Açores  Santa Cruz da Graciosa      1
          Viseu                       Santa Comba Dão             1
          Castelo Branco              Belmonte                    1
Name: count, Length: 194, dtype: int64

In [28]:
subset_plot = (
    subset.dropna(subset=["district"])  # remove missing districts
      .groupby("district")
      .agg(
          n_contracts=("idcontrato", "count"),
          n_adjudicantes=("contribuinte_adjudicante", "nunique"),
          n_adjudicatarios=("contribuinte_adjudicatarios", "nunique")
      )
      .reset_index()
)

In [29]:
fig = go.Figure()

# --- traces ---
part01 = subset_plot.sort_values("n_contracts", ascending=False)
fig.add_trace(go.Bar(
    x=part01["district"],
    y=part01["n_contracts"],
    name="Contracts",
    visible=True  # default visible
))

part02 = subset_plot.sort_values("n_adjudicantes", ascending=False)
fig.add_trace(go.Bar(
    x=part02["district"],
    y=part02["n_adjudicantes"],
    name="Adjudicantes",
    visible=False
))

part03 = subset_plot.sort_values("n_adjudicatarios", ascending=False)
fig.add_trace(go.Bar(
    x=part03["district"],
    y=part03["n_adjudicatarios"],
    name="Adjudicatarios",
    visible=False
))

# --- dropdown ---
fig.update_layout(
    updatemenus=[
        dict(
            buttons=[
                dict(
                    label="Contracts",
                    method="update",
                    args=[{"visible": [True, False, False]},
                          {"title": "Number of Contracts"}]
                ),
                dict(
                    label="Adjudicantes",
                    method="update",
                    args=[{"visible": [False, True, False]},
                          {"title": "Number of Adjudicantes"}]
                ),
                dict(
                    label="Adjudicatarios",
                    method="update",
                    args=[{"visible": [False, False, True]},
                          {"title": "Number of Adjudicatarios"}]
                ),
            ],
            direction="down",
            showactive=True
        )
    ]
)

fig.update_layout(
    title="Contracts per District",
    xaxis_title="District",
    yaxis_title="Count",
    xaxis_tickangle=-45
)

fig.show()

In [30]:
subset = subset[subset['district'] == 'Lisboa']
subset.drop(columns=['LocalExecucao', 'country', 'district'], inplace=True)

In [31]:
subset['city'].value_counts()

city
Lisboa                    503
Sintra                     48
Oeiras                     37
Loures                     24
Cascais                    22
Mafra                      11
Alenquer                    5
Amadora                     5
Torres Vedras               4
Odivelas                    4
Cadaval                     4
Vila Franca de Xira         4
Lourinhã                    4
Sobral de Monte Agraço      2
Name: count, dtype: int64

### <font size=6>3.3.7 Price Columns</font> <a class="anchor" id="3.3.7"></a>
  
[Back to TOC](#toc)

In [32]:
# keep idcontrato
price_cols = ['precoBaseProcedimento', 'precoContratual', 'PrecoTotalEfetivo']
box_df = subset[['idcontrato'] + price_cols].copy()

# convert only price columns
box_df[price_cols] = box_df[price_cols].apply(pd.to_numeric, errors='coerce')

# melt while keeping idcontrato
long_df = box_df.melt(
    id_vars='idcontrato',
    var_name='Column',
    value_name='Value'
).dropna()

fig = px.box(
    long_df,
    x='Column',
    y='Value',
    color='Column',
    points='outliers',
    title='Distribution of Price Columns',
    hover_data=['idcontrato'] 
)

fig.update_layout(showlegend=False)
fig.show()

In [33]:
# Count rows with negative contractual price
neg_count = (pd.to_numeric(subset['precoContratual'], errors='coerce') <= 0).sum()
print(f"Rows with precoContratual < 0: {neg_count}")

Rows with precoContratual < 0: 0


In [34]:
# deleting rows where precoContratual is zero or negative, as they are likely errors
subset = subset[subset['precoContratual'] > 0]

In [35]:
subset.info()

<class 'pandas.DataFrame'>
Index: 690 entries, 315 to 291087
Data columns (total 18 columns):
 #   Column                       Non-Null Count  Dtype         
---  ------                       --------------  -----         
 0   idcontrato                   690 non-null    int64         
 1   tipoContrato                 690 non-null    str           
 2   tipoFimContrato              690 non-null    str           
 3   CPV                          690 non-null    str           
 4   adjudicante                  690 non-null    str           
 5   adjudicatarios               690 non-null    str           
 6   concorrentes                 482 non-null    str           
 7   precoBaseProcedimento        690 non-null    float64       
 8   precoContratual              690 non-null    float64       
 9   PrecoTotalEfetivo            690 non-null    float64       
 10  dataDecisaoAdjudicacao       690 non-null    datetime64[us]
 11  dataCelebracaoContrato       690 non-null    datetime64[

In [37]:
# Ensure numeric conversion
subset['precoContratual'] = pd.to_numeric(
    subset['precoContratual'],
    errors='coerce'
)

# Filter rows under 50k
under_50k = subset[subset['precoContratual'] < 100]

# Show result
print(f"Rows with precoContratual < 50,000: {len(under_50k)}")

under_50k[
    [
        'idcontrato',
        'precoContratual',
        'precoBaseProcedimento',
        'PrecoTotalEfetivo',
        'adjudicante',
        'adjudicatarios'
    ]
].head(50)

Rows with precoContratual < 50,000: 6


,idcontrato,precoContratual,precoBaseProcedimento,PrecoTotalEfetivo,adjudicante,adjudicatarios
17492,11236646,37.20,91045.00,37.20,Santa Casa da Misericórdia de Lisboa,Speculum Artigos Medicos S.A
26509,11195180,99.25,35045.01,99.25,Instituto Superior de Economia e Gestão,EDNI - Empresa Distribiudora de Material Info...
91484,11458295,27.00,488161.27,30.00,"Instituto de Informática, I. P.",maquilotus-comercio de produtos de limpeza lda
140778,11614212,74.00,764476.81,74.00,"Instituto de Proteção e Assistência na Doença,...",TOPTONER - Rec. e Comercialização Consumiveis ...
224895,14474801,69.51,324987.86,69.51,Estrutura de Missão para a Extensão da Platafo...,"VODAFONE PORTUGAL - Comunicações Pessoais, S.A."
226629,11904112,30.00,113125.70,30.00,Instituto Nacional de Medicina Legal e Ciência...,"ENZIFARMA-DIAGNOSTICA E FARMACEUTICA, S.A.."


In [ ]:
# Ensure numeric
subset['precoContratual'] = pd.to_numeric(
    subset['precoContratual'],
    errors='coerce'
)

# --- IQR method ---
Q1 = subset['precoContratual'].quantile(0.25)
Q3 = subset['precoContratual'].quantile(0.75)
IQR = Q3 - Q1

lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

# Find outliers
outliers_df = subset[
    (subset['precoContratual'] < lower_bound) |
    (subset['precoContratual'] > upper_bound)
]

print(f"Number of outliers: {len(outliers_df)}")
print(f"Lower bound: {lower_bound:,.2f}")
print(f"Upper bound: {upper_bound:,.2f}")

# Show relevant columns
outliers_df[
    [
        'idcontrato',
        'precoContratual',
        'precoBaseProcedimento',
        'PrecoTotalEfetivo',
        'adjudicante',
        'adjudicatarios'
    ]
].sort_values('precoContratual', ascending=False)

Number of outliers: 75
Lower bound: -116,122.81
Upper bound: 228,260.69


,idcontrato,precoContratual,precoBaseProcedimento,PrecoTotalEfetivo,adjudicante,adjudicatarios
65462,11348685,2999800.00,3498000.00,2999800.00,SPMS - Serviços Partilhados do Ministério da S...,"Paldata, SA"
118450,11536915,2210994.96,2218193.66,2210994.96,Secretaria-Geral do Ministério da Administraçã...,NÓS COMUNICAÇÕES S.A.
103340,11491161,2179937.00,2180452.18,2179937.00,"Instituto do Emprego e Formação Profissional, IP","INETUM ESPAÑA, S.A. - SUCURSAL EM PORTUGAL"
218658,11813599,1785624.10,2902000.00,1785624.10,Secretaria-Geral do MAI,"Warpcom Services, S.A."
8439,11213291,1733139.48,2174600.00,1733139.48,Instituto de Gestão Financeira e Equipamentos ...,"MEO - Serviços de Comunicações e Multimédia, S.A."
...,...,...,...,...,...,...
53487,11307549,245628.48,245628.48,245628.02,Guarda Nacional Republicana,"Costa &amp; Porfírio, Lda"
9349,11196481,243306.00,275000.00,243306.00,"Secretaria-Geral do Ministério do Trabalho, So...",Digibéria Information Technologies S.A.
28212,11292502,235989.88,260000.00,247681.13,Universidade do Minho,"SCHMID – CONSTRUÇÕES, Lda."
154276,11649988,231042.68,233725.79,231042.68,"Agência para a Modernização Administrativa, I. P.","Claranet Portugal, S.A."


In [ ]:
# TODO: histogram of price distribution with log scale, showing outliers in different color
fig = px.histogram(
    subset,
    x='precoContratual',
    nbins=100,
    title='Distribution of Contractual Price',
    labels={'precoContratual': 'Contractual Price', 'count': 'Number of Contracts'},
    log_y=True
)
fig.show()

### <font size=6>3.3.8 CPV</font> <a class="anchor" id="3.3.8"></a>
  
[Back to TOC](#toc)

In [ ]:
#subset['CPV'] = subset['CPV'].astype(str).str.replace('\n', ' | ', regex=True).str.strip()
#subset['CPV'] = subset['CPV'].astype(str).str.replace(r'\s*\d{8}-\d\s*-\s*', ' ', regex=True).str.strip()

In [ ]:
subset['CPV'].value_counts() 

CPV
45453100-8 - Obras de recuperação                                                   27
30200000-1 - Equipamento e material informático                                     22
30213300-8 - Computadores de secretária (desktop computers)                         20
48000000-8 - Pacotes de software e sistemas de informação                           19
30230000-0 - Equipamento informático                                                18
                                                                                    ..
35112000-2 - Equipamento de socorro e segurança                                      1
45261220-2 - Pintura de telhados e outros revestimentos para telhados                1
48710000-8 - Pacote de software para cópia de segurança (back-up) ou recuperação     1
43251000-7 - Pás carregadoras com retroescavadora                                    1
48620000-0 - Sistemas operativos                                                     1
Name: count, Length: 281, dtype: int64

In [ ]:
subset_counts = subset['CPV'].value_counts().reset_index()
subset_counts.columns = ['CPV', 'count']

# Get top 20
top_20 = subset_counts.head(20)

fig = px.bar(
    top_20,
    x='count',
    y='CPV',
    orientation='h',
    title='Top 20 CPVs by Number of Contracts',
    labels={'CPV': 'CPV', 'count': 'Number of Contracts'},
    width=1700,   
    height=800  
)

# Largest bar on top
fig.update_layout(
    yaxis={'categoryorder': 'total ascending'}
)

fig.show()

In [ ]:
# Ensure CPV is string (important)
subset['CPV'] = subset['CPV'].astype(str)

# Extract first 2 digits
subset['cpv_prefix'] = subset['CPV'].str[:2]

# Mapping dictionary (based on your table)
cpv_map = {
    "03": "Agricultura, pesca e silvicultura",
    "09": "Energia e combustíveis",
    "14": "Mineração e metais",
    "15": "Alimentação, bebidas e tabaco",
    "16": "Maquinaria agrícola",
    "18": "Vestuário e acessórios",
    "19": "Têxteis, couro, plástico e borracha",
    "22": "Material impresso",
    "24": "Produtos químicos",
    "30": "Equipamento de escritório e informática",
    "31": "Equipamento elétrico e iluminação",
    "32": "Telecomunicações",
    "33": "Equipamento médico e farmacêutico",
    "34": "Equipamento de transporte",
    "35": "Segurança e defesa",
    "37": "Desporto, jogos e artesanato",
    "38": "Equipamento laboratorial e óptico",
    "39": "Mobiliário e limpeza",
    "41": "Água tratada",
    "42": "Máquinas industriais",
    "43": "Construção e extração",
    "44": "Materiais de construção",
    "45": "Construção",
    "48": "Software e sistemas de informação",
    "50": "Reparação e manutenção",
    "51": "Instalação",
    "55": "Hotelaria e restauração",
    "60": "Transporte",
    "63": "Serviços auxiliares de transporte",
    "64": "Telecomunicações postais",
    "65": "Serviços públicos",
    "66": "Finanças e seguros",
    "70": "Imobiliário",
    "71": "Arquitetura e engenharia",
    "72": "TI e consultoria",
    "73": "Investigação e desenvolvimento",
    "75": "Administração pública e segurança social",
    "76": "Indústria de petróleo e gás",
    "77": "Agricultura e serviços relacionados",
    "79": "Serviços empresariais",
    "80": "Educação",
    "85": "Saúde e ação social",
    "90": "Ambiente e resíduos",
    "92": "Cultura e desporto",
    "98": "Outros serviços"
}

# Create aggregated CPV category
subset['agg_cpv'] = subset['cpv_prefix'].map(cpv_map).fillna("Outros / Não classificado")

In [ ]:
subset_counts = subset['agg_cpv'].value_counts().reset_index()
subset_counts.columns = ['agg_cpv', 'count']

# Get top 20
top_20 = subset_counts.head(20)

fig = px.bar(
    top_20,
    x='count',
    y='agg_cpv',
    orientation='h',
    title='Top 20 CPVs by Number of Contracts',
    labels={'agg_cpv': 'agg_cpv', 'count': 'Number of Contracts'},
    width=1700,   
    height=800  
)

# Largest bar on top
fig.update_layout(
    yaxis={'categoryorder': 'total ascending'}
)

fig.show()

### <font size=6>3.3.1 Adjudicatário & adjudicante Columns</font> <a class="anchor" id="3.3.1"></a>
  
[Back to TOC](#toc)

In [ ]:
## DASHES
# Detect names starting with dashes
mask = (
    subset['adjudicante'].astype(str).str.strip().str.startswith('-')
    |
    subset['adjudicatarios'].astype(str).str.strip().str.startswith('-')
)

bad_rows = subset[mask]

print(f"Rows with malformed entity names: {len(bad_rows)}")

bad_rows[
    [
        'idcontrato',
        'adjudicante',
        'adjudicatarios'
    ]
].head(10)

Rows with malformed entity names: 10


,idcontrato,adjudicante,adjudicatarios
1636,11158132,ISEG - Instituto Superior de Economia e Gestão,- - EBSCO Information Services S.L.U.
63740,11453664,"Direção-Geral do Livro, dos Arquivos e das Bib...",- - NOA GmbH
108272,11624937,Guarda Nacional Republicana,- - Sérgio José Mamede Gonçalves
111103,11529992,Guarda Nacional Republicana,- - MARIO ANDRE GONÇALVES REIS
135315,11614956,Guarda Nacional Republicana,- - Beatriz Soares Ribeiro
146641,11614959,Guarda Nacional Republicana,- - MARIO ANDRE GONÇALVES REIS
163361,11675260,IMPRENSA NACIONAL - CASA DA MOEDA SA,"- - J.VILASECA, SA"
185634,12071697,Guarda Nacional Republicana,- - Mariana de Jesus Gonçalves Mamede Lopes
202685,12054287,Secretaria-Geral do Ministério da Administraçã...,- - Onretrieval Group SL
253680,13267811,IMPRENSA NACIONAL CASA DA MOEDA,"- - J. VILASECA, SA"


In [ ]:
## LEGAL VARIANTS

adg = subset['adjudicante'].astype(str)
adj = subset['adjudicatarios'].astype(str)

all_entities = pd.concat([adg, adj], ignore_index=True)

legal_df = pd.DataFrame({"original": all_entities})
legal_df["clean"] = legal_df["original"].str.lower().str.strip()

legal_patterns = [
    r'\bs\.?a\.?\b',
    r'\bsociedade anonima\b',
    r'\bltda\b',
    r'\bunipessoal ltda\b',
    r'\bs\.?l\.?\b',
    r'\bs\.?a\.?u\.?\b',
    r'\bs\.?l\.?u\.?\b',
    r'\binc\b',
    r'\bllc\b'
]

for pat in legal_patterns:
    legal_df["base"] = legal_df["clean"].str.replace(pat, "", regex=True)

legal_df["base"] = (
    legal_df["base"]
    .str.replace(r'[.,;:()\-]', ' ', regex=True)
    .str.replace(r'\s+', ' ', regex=True)
    .str.strip()
)

legal_groups = (
    legal_df
    .groupby("base")
    .agg(
        n_variants=("original", "nunique"),
        examples=("original", lambda x: list(set(x)))
    )
    .reset_index()
)

legal_duplicates = legal_groups[legal_groups["n_variants"] > 1]

print(f"Companies differing only by legal form: {len(legal_duplicates)}")

legal_dup_df = legal_duplicates.sort_values("n_variants", ascending=False)

legal_dup_df.head(50)

Companies differing only by legal form: 42


,base,n_variants,examples
103,claranet ii solutions s a,5,"[CLARANET II SOLUTIONS, S.A,, Claranet II Solu..."
105,claranet portugal s a,4,"[CLARANET PORTUGAL, S.A, Claranet Portugal, S...."
242,inetum españa s a sucursal em portugal,4,"[INETUM ESPAÑA S.A. - SUCURSAL EM PORTUGAL, IN..."
185,exitus soluções tecnológicas lda,4,"[Exitus, Soluções Tecnológicas, Lda., Exitus -..."
380,nautilus s a,3,"[NAUTILUS, S.A, NAUTILUS S.A., NAUTILUS, S.A.]"
505,smile viagens e turismo unipessoal lda,3,"[SMILE - VIAGENS E TURISMO, UNIPESSOAL LDA, SM..."
24,almeida &amp neves lda,2,"[Almeida &amp; Neves, Lda, ALMEIDA &amp; NEVES..."
18,agência para a modernização administrativa i p,2,"[Agência para a Modernização Administrativa, I..."
54,avvale unipessoal lda,2,"[AVVALE, UNIPESSOAL, LDA, AVVALE, UNIPESSOAL LDA]"
55,axianseu digital solutions s a,2,"[Axianseu Digital Solutions, S.A, AXIANSEU DIG..."


In [ ]:
import unicodedata
import re


def clean_entity(x):
    x = str(x)

    # remove accents
    x = unicodedata.normalize('NFKD', x).encode('ASCII', 'ignore').decode('utf-8')

    # lowercase
    x = x.lower()

    # remove punctuation
    x = re.sub(r'[.,;:()\-]', ' ', x)

    # normalize spaces
    x = re.sub(r'\s+', ' ', x).strip()

    # -------------------------
    # REMOVE LEGAL FORMS
    # -------------------------

    legal_patterns = [
        r'\bsa\b',
        r'\bs a\b',
        r'\bs a u\b',
        r'\bsau\b',
        r'\bs l\b',
        r'\bsl\b',
        r'\bs l u\b',
        r'\bsociedade anonima\b',
        r'\bsoc anonima\b',
        r'\bltda\b',
        r'\bunipessoal ltda\b',
        r'\bunipessoal lda\b',
        r'\blda\b',
        r'\binc\b',
        r'\bllc\b'
    ]

    for pat in legal_patterns:
        x = re.sub(pat, '', x)

    # final cleanup after removals
    x = re.sub(r'\s+', ' ', x).strip()

    return x

subset['adjudicante_clean'] = subset['adjudicante'].apply(clean_entity)
subset['adjudicatarios_clean'] = subset['adjudicatarios'].apply(clean_entity)

In [ ]:
## LEGAL VARIANTS

adg = subset['adjudicante_clean'].astype(str)
adj = subset['adjudicatarios_clean'].astype(str)

all_entities = pd.concat([adg, adj], ignore_index=True)

legal_df = pd.DataFrame({"original": all_entities})
legal_df["clean"] = legal_df["original"].str.lower().str.strip()

legal_patterns = [
    r'\bs\.?a\.?\b',
    r'\bsociedade anonima\b',
    r'\bltda\b',
    r'\bunipessoal ltda\b',
    r'\bs\.?l\.?\b',
    r'\bs\.?a\.?u\.?\b',
    r'\bs\.?l\.?u\.?\b',
    r'\binc\b',
    r'\bllc\b'
]

for pat in legal_patterns:
    legal_df["base"] = legal_df["clean"].str.replace(pat, "", regex=True)

legal_df["base"] = (
    legal_df["base"]
    .str.replace(r'[.,;:()\-]', ' ', regex=True)
    .str.replace(r'\s+', ' ', regex=True)
    .str.strip()
)

legal_groups = (
    legal_df
    .groupby("base")
    .agg(
        n_variants=("original", "nunique"),
        examples=("original", lambda x: list(set(x)))
    )
    .reset_index()
)

legal_duplicates = legal_groups[legal_groups["n_variants"] > 1]

print(f"Companies differing only by legal form: {len(legal_duplicates)}")

legal_dup_df = legal_duplicates.sort_values("n_variants", ascending=False)

legal_dup_df.head(50)

Companies differing only by legal form: 0


,base,n_variants,examples


## <font size=6>**3.4 Final Data**</font> <a class="anchor" id="3.4"></a>
  
[Back to TOC](#toc)

In [ ]:
subset.head(15)

,idcontrato,tipoContrato,tipoFimContrato,CPV,adjudicante,adjudicatarios,concorrentes,precoBaseProcedimento,precoContratual,PrecoTotalEfetivo,...,dataPublicacao,dataFechoContrato,nr_concorrentes,contribuinte_adjudicante,contribuinte_adjudicatarios,city,cpv_prefix,agg_cpv,adjudicante_clean,adjudicatarios_clean
315,11132143,Aquisição de bens móveis,"O cumprimento, a impossibilidade definitiva e ...",44613800-8 - Contentores para resíduos,Município de Alenquer,Sopinal - Indústria de Equipamentos e Contento...,"500276218-SOPINAL, L.da | 500231206-Resopre - ...",39714.70,26308.50,26308.50,...,2025-01-06,2025-09-23,3,501305734,500276218,Alenquer,44,Materiais de construção,municipio de alenquer,sopinal industria de equipamentos e contentores
357,11133293,Aquisição de serviços,"O cumprimento, a impossibilidade definitiva e ...",63510000-7 - Serviços de agências de viagens e...,"Instituto da Mobilidade e dos Transportes, IP","DOT Viagens e Turismo, Lda.","500886113-Raso - Viagens e Turismo, S.A. | 506...",67180.00,67180.00,65980.96,...,2025-01-06,2026-01-27,5,508195446,514862645,Lisboa,63,Serviços auxiliares de transporte,instituto da mobilidade e dos transportes ip,dot viagens e turismo
407,11135590,Aquisição de bens móveis,"O cumprimento, a impossibilidade definitiva e ...",30210000-4 - Máquinas de processamento de dado...,"Edurumos, Educação, L.da","MEO - Serviços de Comunicações e Multimédia, S...",504615947-Meo - Serviços de Comunicações e Mul...,404027.62,20850.02,20333.92,...,2025-01-07,2025-12-19,6,504682687,504615947,Lisboa,30,Equipamento de escritório e informática,edurumos educacao l da,meo servicos de comunicacoes e multimedia s ai...
780,11143100,Aquisição de serviços,"O cumprimento, a impossibilidade definitiva e ...",79341000-6 - Serviços de publicidade,Autoridade Nacional de Segurança Rodoviária,Nova Expressão - Planeamento de Media e Public...,507247914-Media Gate Agência de Meios e Comuni...,404000.00,139900.00,139900.00,...,2025-01-09,2025-03-26,2,600082563,503160300,Oeiras,79,Serviços empresariais,autoridade nacional de seguranca rodoviaria,nova expressao planeamento de media e publicidade
820,11143152,Aquisição de serviços,"O cumprimento, a impossibilidade definitiva e ...",79341000-6 - Serviços de publicidade,Autoridade Nacional de Segurança Rodoviária,Nova Expressão - Planeamento de Media e Public...,507247914-Media Gate Agência de Meios e Comuni...,404000.00,88900.00,88900.00,...,2025-01-09,2025-03-26,2,600082563,503160300,Oeiras,79,Serviços empresariais,autoridade nacional de seguranca rodoviaria,nova expressao planeamento de media e publicidade
860,11144492,Aquisição de serviços,"O cumprimento, a impossibilidade definitiva e ...",90900000-6 - Serviços de limpeza e saneamento,Serviços Sociais da Guarda Nacional Republicana,dragondisplay unipessoal lda,515109509-dragondisplay unipessoal lda | 51079...,134973.00,6338.80,7142.35,...,2025-01-09,2025-12-31,2,501433813,515109509,Lisboa,90,Ambiente e resíduos,servicos sociais da guarda nacional republicana,dragondisplay
861,11144515,Aquisição de serviços,"O cumprimento, a impossibilidade definitiva e ...",90900000-6 - Serviços de limpeza e saneamento,Serviços Sociais da Guarda Nacional Republicana,Interessantequação Produtos Consultoria e Serv...,515109509-dragondisplay unipessoal lda | 51079...,134973.00,120342.50,122008.36,...,2025-01-09,2025-12-31,2,501433813,510798560,Lisboa,90,Ambiente e resíduos,servicos sociais da guarda nacional republicana,interessantequacao produtos consultoria e serv...
1005,11147117,Aquisição de serviços,"O cumprimento, a impossibilidade definitiva e ...",66510000-8 - Serviços de seguros,Centro de Formação Profissional da Industria M...,"GENERALI SEGUROS, SA",NaN,208140.00,127819.65,127819.65,...,2025-01-10,2025-01-31,0,502077352,500940231,Lisboa,66,Finanças e seguros,centro de formacao profissional da industria m...,generali seguros
1139,11147817,Aquisição de serviços,"O cumprimento, a impossibilidade definitiva e ...",72200000-7 - Serviço

In [ ]:
subset.info()

<class 'pandas.DataFrame'>
Index: 690 entries, 315 to 291087
Data columns (total 22 columns):
 #   Column                       Non-Null Count  Dtype         
---  ------                       --------------  -----         
 0   idcontrato                   690 non-null    int64         
 1   tipoContrato                 690 non-null    str           
 2   tipoFimContrato              690 non-null    str           
 3   CPV                          690 non-null    str           
 4   adjudicante                  690 non-null    str           
 5   adjudicatarios               690 non-null    str           
 6   concorrentes                 482 non-null    str           
 7   precoBaseProcedimento        690 non-null    float64       
 8   precoContratual              690 non-null    float64       
 9   PrecoTotalEfetivo            690 non-null    float64       
 10  dataDecisaoAdjudicacao       690 non-null    datetime64[us]
 11  dataCelebracaoContrato       690 non-null    datetime64[

In [ ]:
print("There are {} Public Entities.".format(subset['adjudicante_clean'].nunique()))
print("There are {} Companies.".format(subset['adjudicatarios_clean'].nunique()))

print("So, in total our analysis contains {} Nodes.".format(
    subset['adjudicatarios_clean'].nunique() + 
    subset['adjudicante_clean'].nunique()))

There are 129 Public Entities.
There are 436 Companies.
So, in total our analysis contains 565 Nodes.


In [ ]:
print("Initial Number of Contracts: {}".format(merged_dataset.shape[0]))
print("Number of Contracts Now: {}".format(subset.shape[0]))
print("Percentage of Deleted Contracts: {}%".format(round((1 - (subset.shape[0]/merged_dataset.shape[0]))*100, 2)))

Initial Number of Contracts: 312566
Number of Contracts Now: 690
Percentage of Deleted Contracts: 99.78%


# <font color='#BFD72F' size=6>**4. Export Preprocessed Data**</font> <a class="anchor" id="4"></a>
  
[Back to TOC](#toc)

In [47]:
subset.to_csv("../data/preprocessed_data.csv", index=False)